# Environment / Dependencies Setup
The cell below installs all required Python packages:

In [ ]:
!pip install PyPDF2 python-dotenv langchain-openai requests

## Import Libraries

In [ ]:
import os
import json
import PyPDF2
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import requests

## Configure the OpenRouter API Key and the LLM
The LLM is accessed through OpenRouter, which gives you one API key for many models, including free ones. Sign up at [openrouter.ai](https://openrouter.ai), create a key, and either put it in a `.env` file as `OPENROUTER_API_KEY=...`, export it, or paste it when prompted by the cell below.

In [ ]:
load_dotenv(".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

print("OpenRouter key loaded.")

The model is a reasoning model, so `extra_body={"reasoning": {"enabled": False}}` turns its hidden reasoning off. Without it, the model can spend the whole `max_tokens` budget thinking and return an empty answer.

In [ ]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.0,
    max_tokens=8000,
    extra_body={"reasoning": {"enabled": False}},
)

### Step 1 — Download and Read the Document
The PDF is downloaded directly from a link and saved locally, then every page's text is pulled out into one block.

In [ ]:
# The URL for the NASA Sun Fact Sheet
pdf_url = "https://radiojove.gsfc.nasa.gov/education/educationalcd/Posters&Fliers/FactSheets/SunFactSheet.pdf"

# Save it one level up in your root directory
local_pdf_path = "data/SunFactSheet.pdf"

os.makedirs("data", exist_ok=True)

print("Downloading PDF...")
try:
    response = requests.get(pdf_url, timeout=30)
    response.raise_for_status()

    # Write the binary content to the file
    with open(local_pdf_path, "wb") as f:
        f.write(response.content)

    print(f"Success! PDF saved locally at: {local_pdf_path}")
except requests.RequestException:
    # The NASA host is sometimes unreachable; reuse the copy already in data/ if there is one
    if not os.path.exists(local_pdf_path):
        raise
    print(f"Download failed - using the existing copy at: {local_pdf_path}")

# Extract text
reader = PyPDF2.PdfReader(local_pdf_path)
raw_text = ""
for page in reader.pages:
    raw_text += page.extract_text() + "\n"

print("Text extraction complete! Ready for AI processing.")

Success: Found the document at '../SunFactSheet.pdf'
Text extraction complete! Ready for AI processing.


### Step 2 — Extract Facts with the LLM
This is the heart of the knowledge-base-building half of the lab. It's easiest to think of it in three parts:

In [ ]:
system_prompt = """
You are a data extraction assistant. Read the text below and extract every distinct technical fact or value present in the source.
Do not limit yourself to a fixed number of facts — extract all of them, no matter how many there are.
Use the same wording as the source text for each fact — do not paraphrase or rename values.
Each fact must have its own separate concept entry, even if two facts are about a related topic.
Keep each "content" field short — 1 to 2 sentences maximum, stating only the fact and its value.
Respond in JSON only, matching this format:
{
  "concepts": [
    {
      "filename": "concept_name.md",
      "type": "concept",
      "title": "Human Readable Title",
      "tags": ["tag1", "tag2"],
      "description": "A single sentence summary",
      "content": "The full detailed explanation in markdown format."
    }
  ]
}
Output ONLY the JSON. No extra text, no markdown formatting blocks.
"""

Then the request, and the reply-reading half: strip any code fence the model wrapped its JSON in, parse it, and fall back to an empty list rather than crashing if it came back malformed.

In [ ]:
print("Sending document to the LLM for extraction...")

prompt = f"{system_prompt}\n\nDocument Text:\n{raw_text}"
response = llm.invoke(prompt)

raw_json = response.content.strip()

# Clean up potential markdown formatting wrapping the JSON
if raw_json.startswith("```"):
    raw_json = raw_json.split("```")[1]
    if raw_json.startswith("json"):
        raw_json = raw_json[4:]
    raw_json = raw_json.strip()

try:
    structured_data = json.loads(raw_json)
    concepts = structured_data.get("concepts", [])
    print(f"Success! AI identified {len(concepts)} distinct concepts.")
except json.JSONDecodeError:
    print("Error: The model's JSON response was cut off or malformed.")
    print("Try reducing the amount of source text, or check the raw output below:")
    print(raw_json[-500:])
    concepts = []

Sending document to Groq for extraction...
Success! AI identified 66 distinct concepts.


### Step 3 — Build the Files and the Index
This step happens in two parts: first the output folder and index file are prepared, then the files are actually written.

In [ ]:
# Prepare the output directory
output_dir = "output_wiki"
os.makedirs(output_dir, exist_ok=True)

index_path = os.path.join(output_dir, "index.md")

# Create master index if it doesn't exist
if not os.path.exists(index_path):
    with open(index_path, "w", encoding="utf-8") as f:
        f.write("# Master Index\n\n")

print(f"Output directory ready at: {output_dir}")

Output directory ready at: ../output_wiki


**Part 2 — Writing the files:**

In [ ]:
for concept in concepts:
    filename = concept['filename'].replace(" ", "_").lower()
    if not filename.endswith('.md'):
        filename += '.md'

    file_path = os.path.join(output_dir, filename)

    okf_content = f"""type: {concept['type']}
title: {concept['title']}
tags: {concept['tags']}
description: {concept['description']}
---
# {concept['title']}

{concept['content']}
"""

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(okf_content)

    print(f"Created: {filename}")

    with open(index_path, "a", encoding="utf-8") as f:
        f.write(f"- **{filename}**: {concept['description']}\n")

Created: sun_mass.md
Created: earth_mass.md
Created: sun_to_earth_mass_ratio.md
Created: sun_gm.md
Created: earth_gm.md
Created: sun_to_earth_gm_ratio.md
Created: sun_volume.md
Created: earth_volume.md
Created: sun_to_earth_volume_ratio.md
Created: sun_volumetric_mean_radius.md
Created: earth_volumetric_mean_radius.md
Created: sun_to_earth_volumetric_mean_radius_ratio.md
Created: sun_mean_density.md
Created: earth_mean_density.md
Created: sun_to_earth_mean_density_ratio.md
Created: sun_surface_gravity.md
Created: earth_surface_gravity.md
Created: sun_to_earth_surface_gravity_ratio.md
Created: sun_escape_velocity.md
Created: earth_escape_velocity.md
Created: sun_to_earth_escape_velocity_ratio.md
Created: sun_ellipticity.md
Created: earth_ellipticity.md
Created: sun_to_earth_ellipticity_ratio.md
Created: sun_moment_of_inertia.md
Created: earth_moment_of_inertia.md
Created: sun_to_earth_moment_of_inertia_ratio.md
Created: sun_visual_magnitude.md
Created: earth_visual_magnitude.md
Created:

### Step 4 — Define the Question

In [27]:
# Set the question you want to ask based on the PDF data
user_query = "What is the magnetic field strength of the Sun's polar field, its sunspots, and its prominences?"

print(f"Question: '{user_query}'")

Question: 'What is the magnetic field strength of the Sun's polar field, its sunspots, and its prominences?'


### Step 5 — Ask the LLM Which Files Are Relevant
This step doesn't try to answer the question yet — it's just figuring out where to look.

In [ ]:
index_path = "output_wiki/index.md"
with open(index_path, "r", encoding="utf-8") as f:
    index_content = f.read()

index_system_prompt = """
You are a retrieval assistant. Read the provided Table of Contents and select the files needed to answer the user's question.
You MUST respond in strict JSON format matching this schema:
{
  "files_to_read": ["filename1.md", "filename2.md"]
}
Output ONLY the JSON. No extra text, no markdown code block formatting.
"""

print("Phase 1: Asking the LLM to review index.md and select relevant files...")

prompt_1 = f"""{index_system_prompt}

Table of Contents:
{index_content}

User Question: {user_query}
"""

Phase 1: Asking the LLM to review index.md and select relevant files...


Send it, and unwrap the JSON if the model fenced it:

In [ ]:
response_1 = llm.invoke(prompt_1)
raw_content_1 = response_1.content.strip()

if raw_content_1.startswith("```"):
    raw_content_1 = raw_content_1.split("```")[1]
    if raw_content_1.startswith("json"):
        raw_content_1 = raw_content_1[4:]
    raw_content_1 = raw_content_1.strip()

Then read the file list off the reply. Nothing has been opened or read in full yet — these are just the names the librarian pointed at.

In [ ]:
retrieval_data = json.loads(raw_content_1)
selected_files = retrieval_data.get("files_to_read", [])

print(f"Success! The LLM requested {len(selected_files)} file(s):")
for file in selected_files:
    print(f" - {file}")

Success! The LLM requested 3 file(s):
 - sun_polar_field_magnetic_field_strength.md
 - sun_sunspots_magnetic_field_strength.md
 - sun_prominences_magnetic_field_strength.md


### Step 6 — Read the Selected Files and Generate the Answer
Now that the relevant file(s) have been picked, this step opens them for real and gets an actual answer.

In [ ]:
output_dir = "output_wiki"
loaded_context = ""

for filename in selected_files:
    file_path = os.path.join(output_dir, filename)
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            loaded_context += f"--- START OF {filename} ---\n{f.read()}\n--- END OF {filename} ---\n\n"
    else:
        print(f"Warning: {filename} not found on disk.")

This time the instruction changes: don't just give an answer, show the reasoning behind it too.

In [ ]:
qa_system_prompt = """
You are an explainable AI system answering user questions using ONLY the provided text from the markdown files.

You MUST respond in strict JSON format with these exact keys:
1. "trace_path": A list of natural language sentences explaining your step-by-step logical thinking process.
2. "sources_used": A list of the specific filenames you used to get the answer.
3. "answer": The direct, clear, and concise answer to the user's question.

Output ONLY valid JSON.
"""

print("\nPhase 2: Sending the selected file contents to generate the final answer...")

prompt_2 = f"""{qa_system_prompt}

Provided Context:
{loaded_context}

User Question: {user_query}
"""

Send it and unwrap the reply:

In [ ]:
response_2 = llm.invoke(prompt_2)
raw_content_2 = response_2.content.strip()

if raw_content_2.startswith("```"):
    raw_content_2 = raw_content_2.split("```")[1]
    if raw_content_2.startswith("json"):
        raw_content_2 = raw_content_2[4:]
    raw_content_2 = raw_content_2.strip()

print("Done")

### Step 7 — View the Explainability Trace

In [ ]:
# Extract and display the final explainable answer
final_result = json.loads(raw_content_2)

print("\nEXPLAINABILITY TRACE\n")
for step in final_result.get("trace_path", []):
    print(f"-> {step}")

print("\nSOURCES CITED\n")
for source in final_result.get("sources_used", []):
    print(f"- {source}")

print("\nFINAL ANSWER\n")
print(final_result.get("answer"))


EXPLAINABILITY TRACE

-> The user is asking for the magnetic field strength of the Sun's polar field, sunspots, and prominences.
-> To find the magnetic field strength of the Sun's polar field, we need to look at the information provided in the sun_polar_field_magnetic_field_strength.md file.
-> According to the sun_polar_field_magnetic_field_strength.md file, the magnetic field strength of the Sun's polar field is 1-2 Gauss.
-> To find the magnetic field strength of the Sun's sunspots, we need to look at the information provided in the sun_sunspots_magnetic_field_strength.md file.
-> According to the sun_sunspots_magnetic_field_strength.md file, the magnetic field strength of the Sun's sunspots is 3000 Gauss.
-> To find the magnetic field strength of the Sun's prominences, we need to look at the information provided in the sun_prominences_magnetic_field_strength.md file.
-> According to the sun_prominences_magnetic_field_strength.md file, the magnetic field strength of the Sun's prom